# Setup — Batch & Streaming Lab

Run this once, top to bottom. About 5 minutes, no code required from you.

It creates:

1. Three schemas — `<you>_bronze`, `<you>_silver`, `<you>_gold`.
2. Three volumes — `raw_files`, `checkpoints`, `schemas`.
3. The batch seed data — products, users and a sales CDC feed with daily deltas.
4. The streaming replay files — a fixed set of 12 JSON files, generated deterministically.

**Compute:** serverless. The streaming notebooks use `availableNow` and short
`processingTime` triggers, both of which run on serverless.

In [ ]:
from helpers import utils, setup_schemas, setup_volumes, seed_data, event_stream

## 1 — Your namespace

In [ ]:
config = utils.get_configs()
for key, value in config.items():
    print(f"{key:20s} {value}")

## 2 — Schemas and volumes

Three volumes this time, not one. Auto Loader needs somewhere to keep its checkpoints
(what has been consumed) and its schema location (what the data looked like last time).
Both belong in Unity Catalog rather than on DBFS.

In [ ]:
setup_schemas.create_user_schemas()
print()
setup_volumes.create_volumes()

## 3 — Batch seed data

- `products` and `users` — daily full snapshots.
- `sales` — a day-0 snapshot followed by daily delta files carrying insert / update /
  delete markers in a `status` column. A `region` column appears partway through the
  series. That is your CDC and schema-drift material.

In [ ]:
seed_data.seed_all()

## 4 — Verify

In [ ]:
print("Batch files")
for dataset in seed_data.DATASETS:
    path = f"{config['volume_raw']}/{dataset}"
    print(f"  {dataset:10s} {len(utils.list_files(path))} file(s)")

print("\nStreaming replay files")
print(f"  {len(utils.list_files(event_stream.replay_path()))} file(s) at {event_stream.replay_path()}")

print("\nManifest")
for key, value in manifest.items():
    print(f"  {key:24s} {value}")

---

Setup is done. The two tracks are independent — start with `B01` or `S01`, whichever your
cohort is covering first.